In [ ]:
import numpy as np
import ssqpy
from time import time, sleep

np.set_printoptions(suppress=True)

# Build MPC
ssqpy.setSilentMode()

dt = 0.01
MPC_H = 10

V_WGT = np.array((6e-1, 2.4e-1))
U_WGT = np.array((0.0, 3e-2))
ELBOW_WGT = 200.0
TIP_WGT = 120.0

elbow_height = 0.1
tip_height = 0.2

torque_clip = 0.08

vel_bounds = np.array((30, 25))
torque_bounds = np.array((0.005, torque_clip,))

model = ssqpy.model.Model(
    MPC_H,
    dt,
    urdf_path="acrobot.urdf",
    solver_mode=ssqpy.model.SolverMode.InverseDynamics,
)

nq = model.getnq()
nv = model.getnv()
nu = model.getnu()

vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(model, V_WGT)
u_cost = ssqpy.model.costs.SquaredControlCost(model, U_WGT)
elbow_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "link2", np.array((0.0, 0.0, elbow_height)), ELBOW_WGT
)
tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "tip", np.array((0.0, 0.0, tip_height)), TIP_WGT
)

for k in range(MPC_H):
    model.addCost(k, vel_cost)
    model.addCost(k, u_cost)
    model.addCost(k, elbow_cost)
    model.addCost(k, tip_cost)

model.addCost(MPC_H, vel_cost)
model.addCost(MPC_H, elbow_cost)
model.addCost(MPC_H, tip_cost)

model.finalize(
    custom_velocity_bounds=vel_bounds,
    custom_torque_bounds=torque_bounds,
)

ssqp_params = ssqpy.solvers.ssqpParams()
ssqp_params.tolerance = 1e-2
ssqp_params.qp_solver_type = ssqpy.solvers.QPSolverType.HPIPM

hpipm_params = ssqpy.solvers.hpipmParams()
hpipm_params.tol_comp = 1e-3
hpipm_params.tol_stat = 1e-3
hpipm_params.tol_eq = 1e-3
hpipm_params.tol_ineq = 1e-3
ssqp_params.hpipmParams = hpipm_params

mpc = ssqpy.solvers.MPC(model, ssqp_params, sqp_iters=6, qp_iters=100)

In [ ]:
def wrap_to_reference(theta, reference=np.pi):
    return reference + np.arctan2(
        np.sin(theta - reference), np.cos(theta - reference)
    )

In [ ]:
from cloudpendulumclient.client import Client
from random import choice

user_token = "MY_TOKEN"

CELL_IDS = [201, 202, 203, 204]

Tf = 60.0
N_EXPERIMENTS = 10

client = Client()

# Results collected across all experiments
all_timestamps = []
all_data = []
urls = []

exp_idx = 0
attempt = 0

while exp_idx < N_EXPERIMENTS:
    attempt += 1
    print(f"=== Running experiment {exp_idx + 1}/{N_EXPERIMENTS} (attempt {attempt}) ===")

    t0 = time()
    timestamps = []
    data = []

    session_token, livestream_url = client.start_experiment(
        user_token=user_token,
        experiment_type="DoublePendulum",
        experiment_time=Tf,
        preparation_time=5.0,
        record=True,
        cell_id=choice(CELL_IDS)
    )
    print("Received response from server!")
    print("Session token: ", session_token)
    print("Livestream url: ", livestream_url)

    current_time = 0.0

    start_all = time()
    while (time() - start_all) < Tf:
        start = time()

        mq = client.get_position(session_token)
        mv = client.get_velocity(session_token)
        mt = client.get_torque(session_token)

        try:
            u = mpc.step(np.hstack((mq, mv)))[1].stage(0)
        except RuntimeError:
            u = np.zeros(nv)

        tau = model.inverseDynamics(np.array(mq), np.array(mv), u)
        tau = np.clip(tau, -torque_bounds, torque_bounds)

        try:
            client.set_torque(tau, session_token)
        except RuntimeError:
            break

        # while time() - start < 0.001:
        #     pass

        elapsed = time() - start

        # print(f"Frequency: {1/elapsed:.2f}Hz")
        timestamps.append(time() - t0)
        data.append(
            (
                np.array(
                    (wrap_to_reference(mq[0]), wrap_to_reference(mq[1], 0.0))
                ),
                mv,
                mt,
            )
        )

    url = client.stop_experiment(session_token)

    print("Final state:", mq)

    all_timestamps.append(timestamps)
    all_data.append(data)
    urls.append(url)
    exp_idx += 1

In [ ]:
import pickle

save_path = f"data/swingup_fc_results_{int(time())}.pkl"
with open(save_path, "wb") as f:
    pickle.dump(
        {
            "all_timestamps": all_timestamps,
            "all_data": all_data,
            "urls": urls,
            "dt": dt,
            "Tf": Tf,
            "N_EXPERIMENTS": N_EXPERIMENTS,
        },
        f,
    )
print(f"Saved raw results to {save_path}")

In [ ]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Plotting (done after all control loops have finished, so forwardKinematics
# calls here don't slow down the real-time control loop above)
# ---------------------------------------------------------------------------

def mask_wrap_jumps(x, threshold=np.pi):
    """Insert NaN wherever consecutive samples jump by more than `threshold`,
    so wrapped angles don't get plotted as a vertical line across the range.
    Works column-wise on an (N, n_joints) array."""
    x = x.astype(float).copy()
    jumps = np.abs(np.diff(x, axis=0)) > threshold
    for col in range(x.shape[1]):
        idx = np.where(jumps[:, col])[0]
        x[idx + 1, col] = np.nan
    return x


def angular_distance(angle, target):
    """Shortest wrapped distance between `angle` and `target`, in [0, pi]."""
    return np.abs(np.mod(angle - target + np.pi, 2 * np.pi) - np.pi)


plot_idx = 0
N_PLOTTED = N_EXPERIMENTS

for exp_idx in range(N_EXPERIMENTS):
    timestamps = all_timestamps[exp_idx]
    data = all_data[exp_idx]

    configs = np.array([d[0] for d in data])
    vels = np.array([d[1] for d in data])
    torques = np.array([d[2] for d in data])

    # Tip position computed here, outside the control loop
    tip_positions = np.array(
        [model.forwardKinematics(cfg, "tip")[0][2] for cfg in configs]
    )

    # Wrapped distance-to-target for each joint (q1 -> pi, q2 -> 0)
    dist_q1 = angular_distance(configs[:, 0], np.pi)
    dist_q2 = angular_distance(configs[:, 1], 0.0)

    # Break the plotted line at wrap-around jumps in joint position
    configs_plot = mask_wrap_jumps(configs)

    fig, axes = plt.subplots(5, 1, figsize=(10, 14), sharex=True)
    fig.suptitle(f"Experiment {plot_idx + 1}/{N_PLOTTED}")

    axes[0].axhline(y=0.09, color="r", label="Minimum tip height (0.09)")
    axes[0].plot(timestamps, tip_positions, label="Tip height")
    axes[0].set_ylabel("Tip position")
    axes[0].legend()

    axes[1].plot(timestamps, dist_q1, label="q1 dist. from pi")
    axes[1].plot(timestamps, dist_q2, label="q2 dist. from 0")
    axes[1].set_ylabel("Distance to target (rad)")
    axes[1].legend()

    axes[2].plot(timestamps, configs_plot)
    axes[2].set_ylabel("Joint position")
    axes[2].legend(["q1", "q2"])

    axes[3].plot(timestamps, vels)
    axes[3].set_ylabel("Joint velocity")

    axes[4].plot(timestamps, torques)
    axes[4].set_ylabel("Torque")
    axes[4].set_xlabel("Time (s)")

    plt.tight_layout()
    plt.savefig(f"data/swingup_{plot_idx + 1}.pdf")

    plot_idx += 1

plt.show()

In [ ]:
import urllib.request

for i, url in enumerate(urls):
    urllib.request.urlretrieve(url, f"data/swingup_{i + 1}.flv")